<a href="https://colab.research.google.com/github/robingilbert1844-cmyk/review-analyzer/blob/main/customer_review_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

to install all necessary deep learning and topic modeling libraries

In [2]:
!pip install -q transformers datasets accelerate bertopic sentence-transformers scikit-learn pandas matplotlib seaborn

It creates project directories, performs a stratified train/test split, fits the TF-IDF vectorizer, and saves the data splits so classical ML models can reuse them instantly without recomputation.

In [3]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# 1. Load Data
df = pd.read_csv("cleaned_reviews.csv")
df.dropna(subset=['cleaned_review', 'sentiment'], inplace=True)
print(f"Dataset successfully loaded: {len(df):,} records")

# 2. Stratified Train/Test Split (80% Train, 20% Test)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_review'],
    df['sentiment'],
    test_size=0.20,
    random_state=42,
    stratify=df['sentiment']
)

# 3. Fit TF-IDF Feature Matrix
vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

# 4. Save Artifacts for classical models
joblib.dump(vectorizer, "models/tfidf_vectorizer.pkl")
joblib.dump((X_train, X_test, y_train, y_test), "data/processed_splits.pkl")
print("Vectorization complete and cached to data/processed_splits.pkl")

Dataset successfully loaded: 189,549 records
Vectorization complete and cached to data/processed_splits.pkl


Model 1: Multinomial Naive Bayes

In [4]:
import time
import joblib
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = joblib.load("data/processed_splits.pkl")

print("--- Training: Multinomial Naive Bayes ---")
start = time.time()
model_nb = MultinomialNB(alpha=0.1)
model_nb.fit(X_train, y_train)
elapsed = round(time.time() - start, 2)

preds_nb = model_nb.predict(X_test)
print(f"Training Time: {elapsed}s | Accuracy: {accuracy_score(y_test, preds_nb)*100:.2f}%\n")
print(classification_report(y_test, preds_nb, digits=4))

joblib.dump(model_nb, "models/multinomial_nb.pkl")

--- Training: Multinomial Naive Bayes ---
Training Time: 0.38s | Accuracy: 94.49%

              precision    recall  f1-score   support

    Negative     0.9239    0.9148    0.9194      5165
     Neutral     0.8619    0.5870    0.6984      3114
    Positive     0.9541    0.9877    0.9706     29631

    accuracy                         0.9449     37910
   macro avg     0.9133    0.8299    0.8628     37910
weighted avg     0.9424    0.9449    0.9413     37910



['models/multinomial_nb.pkl']

Model 2: Logistic Regression with Balanced Weights

In [5]:
import time
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = joblib.load("data/processed_splits.pkl")

print("--- Training: Logistic Regression (Balanced) ---")
start = time.time()
model_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_lr.fit(X_train, y_train)
elapsed = round(time.time() - start, 2)

preds_lr = model_lr.predict(X_test)
print(f"Training Time: {elapsed}s | Accuracy: {accuracy_score(y_test, preds_lr)*100:.2f}%\n")
print(classification_report(y_test, preds_lr, digits=4))

joblib.dump(model_lr, "models/logistic_regression.pkl")

--- Training: Logistic Regression (Balanced) ---
Training Time: 12.33s | Accuracy: 92.25%

              precision    recall  f1-score   support

    Negative     0.9306    0.9417    0.9361      5165
     Neutral     0.5436    0.9114    0.6810      3114
    Positive     0.9930    0.9203    0.9553     29631

    accuracy                         0.9225     37910
   macro avg     0.8224    0.9245    0.8575     37910
weighted avg     0.9476    0.9225    0.9301     37910



['models/logistic_regression.pkl']

Model 3: Linear Support Vector Classifier

In [6]:
import time
import joblib
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = joblib.load("data/processed_splits.pkl")

print("--- Training: Linear SVC (Balanced) ---")
start = time.time()
model_svc = LinearSVC(class_weight='balanced', random_state=42, max_iter=2500)
model_svc.fit(X_train, y_train)
elapsed = round(time.time() - start, 2)

preds_svc = model_svc.predict(X_test)
print(f"Training Time: {elapsed}s | Accuracy: {accuracy_score(y_test, preds_svc)*100:.2f}%\n")
print(classification_report(y_test, preds_svc, digits=4))

joblib.dump(model_svc, "models/linear_svc.pkl")

--- Training: Linear SVC (Balanced) ---
Training Time: 7.94s | Accuracy: 95.62%

              precision    recall  f1-score   support

    Negative     0.9435    0.9373    0.9404      5165
     Neutral     0.7725    0.8060    0.7889      3114
    Positive     0.9786    0.9753    0.9770     29631

    accuracy                         0.9562     37910
   macro avg     0.8982    0.9062    0.9021     37910
weighted avg     0.9569    0.9562    0.9565     37910



['models/linear_svc.pkl']

Model 4: Stochastic Gradient Descent Classifier

In [7]:
import time
import joblib
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score

X_train, X_test, y_train, y_test = joblib.load("data/processed_splits.pkl")

print("--- Training: SGD Classifier ---")
start = time.time()
model_sgd = SGDClassifier(loss='hinge', class_weight='balanced', random_state=42, max_iter=1500)
model_sgd.fit(X_train, y_train)
elapsed = round(time.time() - start, 2)

preds_sgd = model_sgd.predict(X_test)
print(f"Training Time: {elapsed}s | Accuracy: {accuracy_score(y_test, preds_sgd)*100:.2f}%\n")
print(classification_report(y_test, preds_sgd, digits=4))

joblib.dump(model_sgd, "models/sgd_classifier.pkl")

--- Training: SGD Classifier ---
Training Time: 0.56s | Accuracy: 95.77%

              precision    recall  f1-score   support

    Negative     0.9436    0.9386    0.9411      5165
     Neutral     0.8019    0.7916    0.7967      3114
    Positive     0.9763    0.9785    0.9774     29631

    accuracy                         0.9577     37910
   macro avg     0.9072    0.9029    0.9051     37910
weighted avg     0.9575    0.9577    0.9576     37910



['models/sgd_classifier.pkl']

Deep Transformer Supervised Classifier (DistilBERT)

In [8]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

df = pd.read_csv("cleaned_reviews.csv").dropna(subset=['cleaned_review', 'sentiment'])

label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df['label'] = df['sentiment'].map(label2id)

# Stratified balanced subset for stable GPU fine-tuning
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_sub = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 4000), random_state=42))
test_sub = test_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_func(batch):
    return tokenizer(batch['cleaned_review'], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_sub).map(tokenize_func, batched=True)
test_ds = Dataset.from_pandas(test_sub).map(tokenize_func, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1, "precision": p, "recall": r}

training_args = TrainingArguments(
    output_dir="./distilbert_checkpoints",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

print("--- Fine-Tuning DistilBERT on GPU ---")
trainer.train()
metrics = trainer.evaluate()
print("\nFinal DistilBERT Evaluation Metrics:", metrics)

model.save_pretrained("models/distilbert_sentiment_model")
tokenizer.save_pretrained("models/distilbert_sentiment_model")
print("Saved DistilBERT model to models/distilbert_sentiment_model")

/tmp/ipykernel_3002/3467332328.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_sub = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 4000), random_state=42))
/tmp/ipykernel_3002/3467332328.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_sub = test_df.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.191312,0.182310,2,0.926333,0.926884,0.928737,0.926333



Final DistilBERT Evaluation Metrics: {'eval_loss': 0.18231026828289032, 'eval_accuracy': 0.9263333333333333, 'eval_f1': 0.9268835845625739, 'eval_precision': 0.9287366501985387, 'eval_recall': 0.9263333333333333}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved DistilBERT model to models/distilbert_sentiment_model


 Unsupervised Theme Extraction (LDA & BERTopic)
Extract recurring complaint themes (shipping, defective items, build quality) from the negative reviews without manual labels.

Method A: Latent Dirichlet Allocation (LDA)

In [9]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

df = pd.read_csv("cleaned_reviews.csv")
neg_reviews = df[df['sentiment'] == 'Negative']['cleaned_review'].dropna()

tf_vec = CountVectorizer(max_features=5000, max_df=0.85, min_df=5, stop_words='english')
dtm = tf_vec.fit_transform(neg_reviews)

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(dtm)

print("--- Discovered LDA Complaint Themes ---")
words = tf_vec.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    top_words = [words[i] for i in topic.argsort()[:-9:-1]]
    print(f"Theme {idx+1}: {', '.join(top_words)}")

joblib.dump((lda, tf_vec), "models/lda_topic_model.pkl")

--- Discovered LDA Complaint Themes ---
Theme 1: product, worst, bad, experience, expectations, meet, quality, terrible
Theme 2: better, product, expected, good, way, water, use, working
Theme 3: product, flipkart, buy, dont, worst, small, got, return
Theme 4: quality, nan, bad, good, disappointed, poor, product, utterly
Theme 5: money, waste, dont, rubbish, absolute, working, buy, bad


['models/lda_topic_model.pkl']

Method B: BERTopic (Semantic Transformer Topic Modeling)

In [10]:
import pandas as pd
from bertopic import BERTopic

df = pd.read_csv("cleaned_reviews.csv")
neg_samples = df[df['sentiment'] == 'Negative']['cleaned_review'].dropna().sample(
    n=min(len(df[df['sentiment'] == 'Negative']), 12000),
    random_state=42
).tolist()

topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2", min_topic_size=30, verbose=True)
topics, probs = topic_model.fit_transform(neg_samples)

print("--- Top Operational Complaint Themes (BERTopic) ---")
display(topic_model.get_topic_info().head(10))

topic_model.save("models/bertopic_model", serialization="safetensors", save_ctfidf=True)
print("Saved BERTopic model to models/bertopic_model")

2026-09-02 16:28:39,790 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/375 [00:00<?, ?it/s]

2026-09-02 16:28:48,083 - BERTopic - Embedding - Completed ✓
2026-09-02 16:28:48,085 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-02 16:29:19,199 - BERTopic - Dimensionality - Completed ✓
2026-09-02 16:29:19,203 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-02 16:29:19,696 - BERTopic - Cluster - Completed ✓
2026-09-02 16:29:19,705 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-02 16:29:19,847 - BERTopic - Representation - Completed ✓


--- Top Operational Complaint Themes (BERTopic) ---


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2989,-1_working_nan_buy_product,"[working, nan, buy, product, meet, expectation...","[poor product bad dont waste money, poor bad p..."
1,0,323,0_utterly_disappointed_slightly_waist,"[utterly, disappointed, slightly, waist, buy, ...","[utterly disappointed bad product, utterly dis..."
2,1,320,1_small_size_image_large,"[small, size, image, large, length, slightly, ...","[good small size, good small size, good small ..."
3,2,266,2_wifi_router_connect_device,"[wifi, router, connect, device, bluetooth, con...","[waste money wifi, waste money wifi working, h..."
4,3,223,3_months_stopped_month_working,"[months, stopped, month, working, days, worked...",[recommended bad product stopped working withi...
5,4,221,4_flipkart_service_never_seller,"[flipkart, service, never, seller, return, del...","[waste money worst experience flipkart, waste ..."
6,5,188,5_wood_table_assemble_wall,"[wood, table, assemble, wall, screws, rods, st...",[absolute rubbish screws strength wall mountin...
7,6,160,6_cloth_fabric_stitching_nan,"[cloth, fabric, stitching, nan, shirt, clothes...","[nan product bad poor quality cloth, nan bad q..."
8,7,154,7_experience_ever_worst_buy,"[experience, ever, worst, buy, card, memory, p...","[worst experience ever product bad, worst expe..."
9,8,153,8_nan_pant_cheap_low,"[nan, pant, cheap, low, products, quality, buy...","[nan product bad quality, nan product bad qual..."


Saved BERTopic model to models/bertopic_model


Download All Model Assets for Your Flask App

In [11]:
import shutil
from google.colab import files

shutil.make_archive("all_trained_models", 'zip', "models")
files.download("all_trained_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>